# Week 4 Tutorial: Prompt Iteration and Error Analysis

This notebook shows a simple way to test prompt versions across three LLM endpoints.

Goal: choose robust prompts and document failure modes for each model.

## Step 1: Imports

What this does: imports libraries for data processing, prompt parsing, and model calls.

Why it matters: this keeps each later step focused on logic, not setup.

Principle: set up your toolbox first so experiments are repeatable and easy to debug.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Load Week 3 test data

What this does: loads the Week 3 held-out split and defines the allowed label set.

Why it matters: prompt comparison is only meaningful when all versions are tested on the same target and data.

Principle: control your dataset before comparing methods, otherwise comparisons are not fair.

In [2]:
ROOT = Path('..').resolve()
WEEK3_TEST_CSV = 'week3_artifacts/test_split.csv'

ID_COL = 'sample_id'
TEXT_COL = 'text'
LABEL_COL = 'label'

MODEL_ENDPOINTS = [
    {'label': 'qwen2.5-7b', 'model_name': 'Qwen2.5-7B', 'host': 'localhost', 'port': 8000},
    {'label': 'qwen2.5-7b-instruct', 'model_name': 'Qwen2.5-7B-Instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'qwen2.5-coder-7b-instruct', 'model_name': 'Qwen2.5-Coder-7B-Instruct', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'

PROMPT_DIR = ROOT / 'Prompts'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(WEEK3_TEST_CSV)[[ID_COL, TEXT_COL, LABEL_COL]].copy()
label_values = sorted(df[LABEL_COL].astype(str).unique().tolist())
label_set = set(label_values)

print(f'Rows available: {len(df)}')
print(f'Labels: {label_values}')
display(df.head())

Rows available: 40
Labels: ['high_risk', 'low_risk', 'moderate_risk']


,sample_id,text,label
0,1,Rapidly dividing cells with high mitotic index...,high_risk
1,2,The culture showed stable morphology and low i...,low_risk
2,3,Mild elevation in cytokines was detected after...,moderate_risk
3,4,Gene panel indicates multiple pathogenic varia...,high_risk
4,5,Protein expression remained within baseline an...,low_risk


## Step 3: Define prompt versions

What this does: creates alternative prompt templates (baseline and few-shot) and one parser for all outputs.

Why it matters: you can test formatting and accuracy differences while keeping evaluation logic constant.

Principle: change one variable at a time. Here, prompt text changes while parser and scoring stay fixed.

In [3]:
few_shot_samples = [
    {
        'input_text': 'Example from your domain: baseline profile with no warning signals.',
        'output_json': {'prediction': label_values[0], 'confidence': 0.82}
    },
    {
        'input_text': 'Example from your domain: strong evidence of severe progression.',
        'output_json': {'prediction': label_values[-1], 'confidence': 0.91}
    }
]

def build_prompt(text, labels, version='v1', few_shot=None):
    labels_txt = json.dumps(labels)

    if version == 'v1':
        return (
            'You are solving a classification task.\n'
            f'Allowed labels: {labels_txt}\n'
            'Return ONLY JSON: {\"prediction\": \"<label>\", \"confidence\": <0_to_1_float>}\n'
            'No extra text.\n\n'
            f'Text:\n{text}'
        )

    if version == 'v2_few_shot':
        prompt_lines = [
            'You are solving a classification task.',
            f'Allowed labels: {labels_txt}',
            'Follow these examples and output JSON only.'
        ]
        if few_shot:
            for i, ex in enumerate(few_shot, start=1):
                prompt_lines.append(f'Example {i} input: {ex["input_text"]}')
                prompt_lines.append(f'Example {i} output: {json.dumps(ex["output_json"])}')
        prompt_lines.extend([
            'Output schema: {\"prediction\": \"<label>\", \"confidence\": <0_to_1_float>}',
            '',
            'Text:',
            str(text)
        ])
        return '\n'.join(prompt_lines)

    raise ValueError(f'Unknown version: {version}')

def parse_response(raw_text, valid_labels):
    if not isinstance(raw_text, str) or not raw_text.strip():
        return None, np.nan, False, 'empty_response'

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        return None, np.nan, False, 'invalid_json'

    pred = str(payload.get('prediction', '')).strip()
    if pred not in valid_labels:
        return None, np.nan, False, 'invalid_label'

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    return pred, conf, True, None

## Step 4: Evaluate prompt versions on a small subset

What this does: runs each prompt version on the same subset and computes parse success and accuracy.

Why it matters: small-scale testing is faster and helps pick a strong prompt before full benchmarking.

Principle: iterate quickly on a representative subset, then scale to the full test set once stable.

WARNING: Nested for loops + LLM queries = death by boredom and kernel crashes from time outs. In your actual testing keep subsets small and/or query one model at a time.

In [4]:
SYSTEM_PROMPT = 'You follow output formatting instructions exactly.'

clients = {
    cfg['label']: OpenAI(
        api_key=API_KEY,
        base_url=f"http://{cfg['host']}:{cfg['port']}/v1"
    )
    for cfg in MODEL_ENDPOINTS
}

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = clients[endpoint_cfg['label']]
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

subset = df.sample(n=min(5, len(df)), random_state=7).reset_index(drop=True)
versions = ['v1', 'v2_few_shot']

records = []
for endpoint_cfg in MODEL_ENDPOINTS:
    for version in versions:
        for _, row in subset.iterrows():
            prompt = build_prompt(
                text=row[TEXT_COL],
                labels=label_values,
                version=version,
                few_shot=few_shot_samples if version == 'v2_few_shot' else None
            )

            raw = query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0)
            pred, conf, parse_ok, parse_error = parse_response(raw, label_set)

            records.append({
                ID_COL: row[ID_COL],
                'model_label': endpoint_cfg['label'],
                'model_name': endpoint_cfg['model_name'],
                'endpoint_port': endpoint_cfg['port'],
                'version': version,
                'ground_truth': str(row[LABEL_COL]),
                'prediction': pred,
                'confidence': conf,
                'parse_ok': parse_ok,
                'parse_error': parse_error,
                'raw_response': raw
            })

results = pd.DataFrame(records)
results['is_correct'] = (results['ground_truth'] == results['prediction'])

summary = results.groupby(['model_label', 'model_name', 'endpoint_port', 'version'], as_index=False).agg(
    parse_success_rate=('parse_ok', 'mean'),
    accuracy=('is_correct', 'mean'),
    rows=('is_correct', 'size')
)
summary = summary.sort_values(['model_label', 'parse_success_rate', 'accuracy'], ascending=[True, False, False])
display(summary)

,model_label,model_name,endpoint_port,version,parse_success_rate,accuracy,rows
1,qwen2.5-7b,Qwen2.5-7B,8000,v2_few_shot,1.0,0.8,5
0,qwen2.5-7b,Qwen2.5-7B,8000,v1,0.4,0.4,5
2,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,v1,1.0,0.8,5
3,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,v2_few_shot,1.0,0.8,5
4,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,v1,1.0,1.0,5
5,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,v2_few_shot,1.0,0.6,5


## Step 5: Inspect failures

What this does: shows failed parses with raw responses and error labels.

Why it matters: this evidence is what you use for the required Week 4 error analysis.

Principle: error analysis should be example-driven, not only metric-driven.

In [5]:
failures = results[~results['parse_ok']].copy()
display(failures[['version', 'raw_response', 'parse_error']].head(5))
print(f'Total failures: {len(failures)} out of {len(results)}')

,version,raw_response,parse_error
0,v1,"{""prediction"": ""moderate_risk"", ""confidence"": ...",invalid_json
1,v1,"{""prediction"": ""low_risk"", ""confidence"": 0.95}...",invalid_json
3,v1,"{""prediction"": ""high_risk"", ""confidence"": 0.95...",invalid_json


Total failures: 3 out of 30


## Step 6: Save final prompt files

What this does: writes your selected prompt template and few-shot examples to the submission folder.

Why it matters: prompt artifacts make your workflow auditable and reproducible for judges.

Principle: version your prompts like code. Prompt files are part of the experiment definition.

In [6]:
best_by_model = (
    summary.sort_values(['model_label', 'parse_success_rate', 'accuracy'], ascending=[True, False, False])
    .groupby('model_label', as_index=False)
    .first()
    [['model_label', 'model_name', 'endpoint_port', 'version']]
 )
display(best_by_model)

# Save one canonical template + few-shot file for submission compatibility.
overall_best_version = summary.sort_values(['parse_success_rate', 'accuracy'], ascending=False).iloc[0]['version']
final_prompt_example = build_prompt(
    text='[replace with representative text]',
    labels=label_values,
    version=overall_best_version,
    few_shot=few_shot_samples if overall_best_version == 'v2_few_shot' else None
)
prompt_template_path = PROMPT_DIR / 'prompt_template.txt'
few_shot_path = PROMPT_DIR / 'few_shot_samples.json'
prompt_template_path.write_text(final_prompt_example, encoding='utf-8')
few_shot_path.write_text(json.dumps(few_shot_samples, indent=2), encoding='utf-8')

# Also save model-specific prompt choices.
for _, row in best_by_model.iterrows():
    model_prompt = build_prompt(
        text='[replace with representative text]',
        labels=label_values,
        version=row['version'],
        few_shot=few_shot_samples if row['version'] == 'v2_few_shot' else None
    )
    model_prompt_path = PROMPT_DIR / f"prompt_template_{row['model_label']}.txt"
    model_prompt_path.write_text(model_prompt, encoding='utf-8')
    print(f"Saved: {model_prompt_path}")

print(f'Overall best prompt version: {overall_best_version}')
print(f'Saved: {prompt_template_path}')
print(f'Saved: {few_shot_path}')

,model_label,model_name,endpoint_port,version
0,qwen2.5-7b,Qwen2.5-7B,8000,v2_few_shot
1,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,v1
2,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,v1


Saved: /home/jupyter-abby/Prompts/prompt_template_qwen2.5-7b.txt
Saved: /home/jupyter-abby/Prompts/prompt_template_qwen2.5-7b-instruct.txt
Saved: /home/jupyter-abby/Prompts/prompt_template_qwen2.5-coder-7b-instruct.txt
Overall best prompt version: v1
Saved: /home/jupyter-abby/Prompts/prompt_template.txt
Saved: /home/jupyter-abby/Prompts/few_shot_samples.json


## Step 7: Draft report text

What this does: creates starter language for approach and error-analysis sections.

Why it matters: structured drafting helps teams report results consistently and quickly.

Principle: automate repetitive reporting, then refine with concrete findings from your run.

In [7]:
top_rows = summary.sort_values(['parse_success_rate', 'accuracy'], ascending=False).head(3)
display(top_rows)

approach_report = (
    'We evaluated three LLM endpoints and compared prompt versions on the same Week 3 held-out split subset. '
    'For each model, we selected a prompt based on parse reliability and accuracy. '
    'Outputs were constrained to strict JSON and validated against allowed labels before scoring.'
)

error_analysis = (
    'Common failures included malformed JSON, occasional labels outside the allowed set, and inconsistent formatting across models. '
    'Prompt wording affected compliance and quality differently by model. '
    'Few-shot prompting often improved reliability but did not remove all parsing errors.'
)

print('Approach Report draft:\n')
print(approach_report)
print('\nError Analysis draft:\n')
print(error_analysis)

,model_label,model_name,endpoint_port,version,parse_success_rate,accuracy,rows
4,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002,v1,1.0,1.0,5
1,qwen2.5-7b,Qwen2.5-7B,8000,v2_few_shot,1.0,0.8,5
2,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001,v1,1.0,0.8,5


Approach Report draft:

We evaluated three LLM endpoints and compared prompt versions on the same Week 3 held-out split subset. For each model, we selected a prompt based on parse reliability and accuracy. Outputs were constrained to strict JSON and validated against allowed labels before scoring.

Error Analysis draft:

Common failures included malformed JSON, occasional labels outside the allowed set, and inconsistent formatting across models. Prompt wording affected compliance and quality differently by model. Few-shot prompting often improved reliability but did not remove all parsing errors.


### TRY THIS

- Add a third prompt version
- Test a different subset size
- Add a stricter parser rule and compare failure counts
